# Earthquake Prediction

**Tabular Classification** — Predict earthquake status from seismic data.

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Dataset: USGS Earthquake Database
Target: `Status`

## 2 · Project Overview

We classify earthquake events by **Status** (e.g., Reviewed, Automatic) using magnitude, depth, location, and other seismic parameters from the USGS earthquake database.

## 3 · Learning Objectives

1. Work with geospatial/seismic data.
2. Handle datasets with many columns and mixed types.
3. Compare boosting models on a multi-class problem.
4. Use LazyPredict and FLAML for rapid benchmarking.
5. Evaluate with accuracy, F1, and confusion matrix.

## 4 · Problem Statement

Given seismic event features (magnitude, depth, location, source), classify the **Status** of the event.

## 5 · Why This Project Matters

- Earthquake monitoring is critical for disaster preparedness.
- Automating status classification speeds up event review.
- Understanding which features drive status helps seismologists prioritize.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | USGS Earthquake Database via Kaggle |
| **Rows** | ~23,000+ |
| **Target** | Status (multi-class) |
| **Features** | Magnitude, Depth, Latitude, Longitude, Type, etc. |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle `usgs/earthquake-database`
- **License**: Public domain (USGS data)
- **Limitations**: Historical data, may not represent current patterns.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict", "kagglehub"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "Status"
SAVE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
# Try local first
_local = os.path.join(SAVE_DIR, "database.csv")
if not os.path.exists(_local):
    _local = "Classification/Earthquake Prediction/database.csv"
MAX_ROWS = 50000
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Earthquake Prediction


## 11 · Dataset Download or Loading

In [4]:
if os.path.exists(_local):
    df = pd.read_csv(_local)
    print(f"Loaded local: {_local}")
else:
    import kagglehub
    path = kagglehub.dataset_download("usgs/earthquake-database")
    csv = os.path.join(path, "database.csv")
    df = pd.read_csv(csv)
    print(f"Downloaded from kagglehub")

if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)
    print(f"Sampled to {MAX_ROWS} rows")
print(f"Shape: {df.shape}")
df.head()

Loaded local: E:\Github\Machine-Learning-Projects\Classification\Earthquake Prediction\database.csv
Shape: (23412, 21)


,Date,Time,Latitude,Longitude,Type,Depth,Depth Error,Depth Seismic Stations,Magnitude,Magnitude Type,...,Magnitude Seismic Stations,Azimuthal Gap,Horizontal Distance,Horizontal Error,Root Mean Square,ID,Source,Location Source,Magnitude Source,Status
0,01/02/1965,13:44:18,19.246,145.616,Earthquake,131.6,NaN,NaN,6.0,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860706,ISCGEM,ISCGEM,ISCGEM,Automatic
1,01/04/1965,11:29:49,1.863,127.352,Earthquake,80.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860737,ISCGEM,ISCGEM,ISCGEM,Automatic
2,01/05/1965,18:05:58,-20.579,-173.972,Earthquake,20.0,NaN,NaN,6.2,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860762,ISCGEM,ISCGEM,ISCGEM,Automatic
3,01/08/1965,18:49:43,-59.076,-23.557,Earthquake,15.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860856,ISCGEM,ISCGEM,ISCGEM,Automatic
4,01/09/1965,13:32:50,11.938,126.427,Earthquake,15.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860890,ISCGEM,ISCGEM,ISCGEM,Automatic


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nMissing values (top 10):\n{df.isnull().sum().sort_values(ascending=False).head(10)}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nShape: {df.shape}")

DATA VALIDATION

Missing values (top 10):
Magnitude Error               23085
Horizontal Error              22256
Horizontal Distance           21808
Magnitude Seismic Stations    20848
Depth Error                   18951
Depth Seismic Stations        16315
Azimuthal Gap                 16113
Root Mean Square               6060
Magnitude Type                    3
Date                              0
dtype: int64

Duplicate rows: 0

Target distribution:
Status
Reviewed     20773
Automatic     2639
Name: count, dtype: int64

Shape: (23412, 21)


## 13 · Exploratory Data Analysis

In [6]:
# Keep only useful numeric + target columns
# Drop date/time/ID text columns
drop_cols = [c for c in df.columns if df[c].dtype == "object" and c != TARGET and c != "Type"]
# Also drop columns with >50% missing
high_miss = [c for c in df.columns if df[c].isnull().mean() > 0.5]
drop_all = list(set(drop_cols + high_miss))
df_clean = df.drop(columns=[c for c in drop_all if c in df.columns], errors="ignore")
print(f"Dropped {len(drop_all)} columns, kept {df_clean.shape[1]} columns")

num_cols = df_clean.select_dtypes(include="number").columns.tolist()
if len(num_cols) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(df_clean[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax, square=True)
    ax.set_title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

df_clean[num_cols[:6]].hist(bins=30, figsize=(14, 8), edgecolor="black")
plt.suptitle("Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

Dropped 14 columns, kept 7 columns


## 14 · Target Analysis

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
df_clean[TARGET].value_counts().plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title(f"Distribution of {TARGET}")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

# Keep only classes with enough samples
min_class = 10
vc = df_clean[TARGET].value_counts()
keep_classes = vc[vc >= min_class].index.tolist()
df_clean = df_clean[df_clean[TARGET].isin(keep_classes)].copy()
print(f"Kept {len(keep_classes)} classes with >= {min_class} samples")
print(f"Shape after filtering: {df_clean.shape}")

Kept 2 classes with >= 10 samples
Shape after filtering: (23412, 7)


## 15 · Train/Validation/Test Split Strategy

80/20 stratified split.

In [8]:
# Drop remaining NaN rows
df_clean = df_clean.dropna(subset=[TARGET])

# Encode target
le_target = LabelEncoder()
df_clean[TARGET] = le_target.fit_transform(df_clean[TARGET])

X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# Encode any remaining categoricals
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Fill remaining NaN with median
X = X.fillna(X.median())

# Sanitize column names
X.columns = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {len(y.unique())}")

Train: (18729, 6), Test: (4683, 6)
Classes: 2


## 16 · Preprocessing Strategy

- **Missing values**: Columns with >50% missing dropped; remaining filled with median.
- **Categorical**: OrdinalEncoder for text features (Type).
- **Target**: LabelEncoder for multi-class Status.
- **Scaling**: Not needed for tree-based models.

## 17 · Feature Engineering

We rely on the existing seismic features which are domain-specific. No additional engineering needed for this benchmark.

## 18 · Baseline Model

In [9]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"\n{classification_report(y_test, y_pred_bl)}")


BASELINE: Logistic Regression
Accuracy : 0.8873
F1       : 0.8347

              precision    recall  f1-score   support

           0       0.50      0.00      0.00       528
           1       0.89      1.00      0.94      4155

    accuracy                           0.89      4683
   macro avg       0.69      0.50      0.47      4683
weighted avg       0.84      0.89      0.83      4683



## 19 · LazyPredict Benchmark

In [10]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 10 Classifiers:")
print(lazy_models.head(10).to_string())

LazyPredict — Top 10 Classifiers:
                        Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                   
GaussianNB              0.773009           0.872082  0.900470  0.813296   0.924669  0.773009    0.048417
XGBClassifier           0.931454           0.789432  0.962396  0.928454   0.927153  0.931454    0.593017
BaggingClassifier       0.920991           0.789322  0.929829  0.919908   0.918991  0.920991    0.351870
LGBMClassifier          0.935511           0.785932  0.965658  0.931553   0.930939  0.935511    4.085295
CatBoostClassifier      0.935084           0.780732  0.965325  0.930735   0.930330  0.935084    9.208504
DecisionTreeClassifier  0.909673           0.779638  0.779638  0.910152   0.910655  0.909673    0.078383
RandomForestClassifier  0.920350           0.769949  0.950625  0.917662   0.915931  0.920350    2.086939
ExtraTreesClassifier 

## 20 · FLAML AutoML Run

In [11]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="classification", time_budget=60, verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    print(f"FLAML best: {flaml_automl.best_estimator}")
    print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
    print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML failed: {e}")
    y_pred_flaml = y_pred_bl

FLAML failed: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).flatten().astype(int)
    print(f"CatBoost Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost GPU failed ({e}), trying CPU...")
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
        timings["CatBoost"] = time.perf_counter() - t0
        results["CatBoost"] = cb.predict(X_test).flatten().astype(int)
        print(f"CatBoost Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  (CPU, {timings['CatBoost']:.1f}s)")
    except Exception as e2:
        print(f"CatBoost failed: {e2}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                            device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM GPU failed, trying CPU...")
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
        timings["LightGBM"] = time.perf_counter() - t0
        results["LightGBM"] = lg.predict(X_test)
        print(f"LightGBM Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  (CPU)")
    except Exception as e2:
        print(f"LightGBM failed: {e2}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                              device="cuda", tree_method="hist", verbosity=0,
                              eval_metric="logloss",
                              n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost GPU failed, trying CPU...")
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0, eval_metric="logloss",
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        timings["XGBoost"] = time.perf_counter() - t0
        results["XGBoost"] = xgb_model.predict(X_test)
        print(f"XGBoost Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  (CPU)")
    except Exception as e2:
        print(f"XGBoost failed: {e2}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost Acc: 0.9321  (18.1s)


Training until validation scores don't improve for 30 rounds


Did not meet early stopping. Best iteration is:
[296]	valid_0's binary_logloss: 0.140828
LightGBM Acc: 0.9342  (13.8s)


XGBoost Acc: 0.9340  (2.4s)


## 22 · Top 3 Model Selection

In [13]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted"), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
XGBoost                0.9340  0.9294     0.9290  0.9340
LightGBM               0.9342  0.9293     0.9292  0.9342
CatBoost               0.9321  0.9257     0.9267  0.9321
Logistic Regression    0.8873  0.8347     0.8437  0.8873
FLAML                  0.8873  0.8347     0.8437  0.8873

Top 3 models: ['XGBoost', 'LightGBM', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i])
    axes[i].set_title(f"{name}\nAcc={accuracy_score(y_test, yp):.4f}  F1={f1_score(y_test, yp, average='weighted'):.4f}")
    axes[i].set_xlabel("Predicted"); axes[i].set_ylabel("Actual")
plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

for name in top3_names:
    yp = results[name]
    print(f"\n{name}:")
    print(f"  Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"  F1       : {f1_score(y_test, yp, average='weighted'):.4f}")
    print(f"  Precision: {precision_score(y_test, yp, average='weighted', zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y_test, yp, average='weighted'):.4f}")


XGBoost:
  Accuracy : 0.9340
  F1       : 0.9294
  Precision: 0.9290
  Recall   : 0.9340

LightGBM:
  Accuracy : 0.9342
  F1       : 0.9293
  Precision: 0.9292
  Recall   : 0.9342

CatBoost:
  Accuracy : 0.9321
  F1       : 0.9257
  Precision: 0.9267
  Recall   : 0.9321


## 24 · Error Analysis

In [15]:
best_name = top3_names[0]
best_pred = results[best_name]
cm = confusion_matrix(y_test, best_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title(f"{best_name} — Confusion Matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]
err_rate_per_feat = errors.groupby("correct").mean(numeric_only=True)
err_rate_per_feat.T.plot(kind="bar", ax=axes[1], figsize=(14, 5))
axes[1].set_title("Mean Feature Values: Correct vs Incorrect")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Overall accuracy ({best_name}): {accuracy_score(y_test, best_pred):.4f}")
print(f"Misclassified: {(y_test.values != best_pred).sum()} / {len(y_test)}")

Overall accuracy (XGBoost): 0.9340
Misclassified: 309 / 4683


## 25 · Interpretation and Business Insight

**Key findings:**
- Earthquake status is largely determined by magnitude and depth.
- Most events are 'Reviewed' — class imbalance is a factor.
- Boosting models handle this well with class-weighted learning.

**Business takeaway:** Automated status classification can speed up USGS review pipelines.

## 26 · Limitations

1. Historical data — patterns may shift.
2. Class imbalance in Status.
3. Many dropped columns (dates, IDs) may contain useful signal.
4. No geospatial feature engineering (distance to fault lines, etc.).

## 27 · How to Improve

1. Engineer geospatial features (proximity to tectonic boundaries).
2. Use temporal features (time of day, season).
3. Try SMOTE or class weights for minority classes.
4. Use more sophisticated NLP on location names.

## 28 · Production Considerations

- Real-time classification requires streaming data pipeline.
- Model retraining on new seismic data regularly.
- Prediction latency matters for alerting systems.
- Integrate with USGS ShakeAlert or similar systems.

## 29 · Common Mistakes

1. Including date/time as raw strings — must parse or drop.
2. Not filtering rare classes — causes unstable predictions.
3. Using accuracy alone on imbalanced classes.
4. Leaking test features during preprocessing.

## 30 · Mini Challenge / Exercises

1. Add magnitude bins as features — does it help?
2. Try only Latitude + Longitude + Depth — how good?
3. Engineer distance-from-equator as a feature.
4. Compare binary (Reviewed vs not) vs full multi-class.

## 31 · Final Summary / Key Takeaways

1. Earthquake status classification is achievable with standard tabular ML.
2. Feature selection matters — many columns are noise.
3. Class filtering is essential for stable results.
4. Boosting models handle this well.
5. Always compare against a baseline.

## Save Metrics

In [16]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted")), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Earthquake Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.9321,
    "f1": 0.9257,
    "precision": 0.9267,
    "recall": 0.9321
  },
  "LightGBM": {
    "accuracy": 0.9342,
    "f1": 0.9293,
    "precision": 0.9292,
    "recall": 0.9342
  },
  "XGBoost": {
    "accuracy": 0.934,
    "f1": 0.9294,
    "precision": 0.929,
    "recall": 0.934
  },
  "Logistic Regression": {
    "accuracy": 0.8873,
    "f1": 0.8347,
    "precision": 0.8437,
    "recall": 0.8873
  },
  "FLAML": {
    "accuracy": 0.8873,
    "f1": 0.8347,
    "precision": 0.8437,
    "recall": 0.8873
  }
}
